# bgm_ace_step

**ACE-Step 1.5** 로 BGM 생성. Stable Audio 3와 비교하기 위한 대조군.

| | |
|---|---|
| 커널 | `ace` |
| 출력 | `outputs/bgm_ace_step/<시나리오>/` |
| 개수 | 시나리오당 페이즈 2 x 후보 3 = 6개 |

태그 기반 제어가 본래 인터페이스라 산문이 아니라 스타일 태그를 넣는다.

## 설치가 까다롭다

- **파이썬 3.11 이상** 필요 (3.10에서는 설치 자체가 거부된다)
- `nano-vllm` 이 PyPI에 없어 **GitHub에서 먼저 설치**해야 한다

아래 1번 셀이 그 순서를 지킨다. 그래도 실패하면 이 노트북은 건너뛰어도 된다 —
`bgm_stable_audio` 만으로 "문화 조건이 반영되는가"라는 질문에는 답이 나온다.


## 1. 커널 확인

설치는 `setup_kernels.ipynb` 가 한다 (`uv sync`).

`pip install git+...` 로는 설치가 **불가능**하다. pyproject 가 linux x86_64 에서
`torch==2.10.0+cu128` 을 요구하는데 그 버전은 PyPI 가 아니라 `[tool.uv.index]` 로
선언된 cu128 인덱스에만 있고, `nano-vllm` 도 `[tool.uv.sources]` 매핑이다.
pip 은 그 설정을 읽지 않는다.

In [ ]:
# ---------------------------------------------------------------------------
# 커널 확인. 설치는 setup_kernels.ipynb 가 담당한다.
#
# 이 셀에 pip install 을 추가하지 말 것. 노트북 4개는 서로 양립 불가능한
# torch/transformers 를 쓰기 때문에, 여기서 다시 깔면 커널 조합이 깨진다.
#   ace  torch 2.10.0+cu128 / transformers >=4.51,<4.58
#   sa3  torch 2.7.1 cu126  / transformers >=5.8
#   flux torch 2.8.0        / transformers ==4.56.1
# ---------------------------------------------------------------------------
import sys, importlib

KERNEL = "ace"
NEED   = ["torch", "soundfile", "acestep"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.ipynb 를 먼저 실행한다.\n"
    f"여기서 pip install 로 때우지 말 것 — 버전 조합이 깨진다."
)
print("\n커널 OK")

## 2. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

# ---- 이 노드에서 유효한 루트를 직접 찾는다 --------------------------------
# 학과 클러스터는 같은 저장소가 노드마다 다른 경로에 마운트된다.
#   로그인 노드(hpcmaster) : /data1/$USER   (= HOME)
#   계산 노드(hpc-stat1)   : /mnt/data1/$USER
# setup_kernels 로 만든 커널이면 PROJ_BASE 가 kernel.json 에 박혀 있고,
# Slurm 으로 돌리면 common.sh 가 환경변수로 넘겨준다. 둘 다 아니면 여기서 찾는다.
def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

# 계산 노드에는 $HOME 이 아예 없다. 두면 캐시/kernelspec 이 전부 깨진다.
if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "bgm_ace_step"
OUT_DIR = BASE / "outputs" / MODEL

# 캐시는 홈이 아니라 대용량 스토리지로. 홈 쿼터면 다운로드 중 os error 28 이 난다.
os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

try:
    for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]),
              pathlib.Path(os.environ["TMPDIR"])):
        p.mkdir(parents=True, exist_ok=True)
except (PermissionError, OSError) as e:
    raise SystemExit(
        f"{BASE} 에 쓸 수 없다 -> {type(e).__name__}: {e}\n"
        f"  이 셀 맨 위에서 지정할 것:\n"
        f"      os.environ['PROJ_BASE'] = '/data1/{USER}/mystery'"
    )

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
# papermill 로 돌리면 cwd 가 /tmp 라 "." 만으로는 못 찾는다.
# PROJ_INPUT(환경변수) 을 먼저 보고, 없으면 후보들을 뒤진다.
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    # 마지막 수단: ROOT 아래를 얕게 훑는다
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  PROJ_INPUT = {os.environ.get('PROJ_INPUT', '(미설정)')}\n"
        f"  cwd        = {pathlib.Path.cwd()}\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

# scenario_id(파일 내부 필드, 예: "12_heungbu")를 키로 쓴다 — 파일명과 무관하게 안정적.
SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    """OUT_DIR/<시나리오>/<...> 경로를 만들고 돌려준다."""
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("ROOT    :", ROOT)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개:")
for k, d in DATA.items():
    print(f"  {k:24s} 장소 {len(d['places'])} / 인물 {len(d['cast'])}")


In [ ]:
import gc, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("torch  :", torch.__version__)
print("device :", DEVICE)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU    :", p.name, "|", round(p.total_memory / 1e9, 1), "GB",
          "| compute capability", f"{p.major}.{p.minor}")
    if (p.major, p.minor) < (8, 9):
        print("  주의: FP8 연산 미지원 세대. FP8 가중치를 요구하는 모델은 못 돌린다.")
else:
    print("  GPU가 없다. 로그인 노드이거나 GPU 할당이 안 된 상태.")


def free():
    """모델을 내린 뒤 호출. 커널을 종료하는 것이 가장 확실하다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        print(f"할당된 VRAM: {used:.2f} GB")


## 3. 태그 조립

`bgm_stable_audio` 와 같은 조건에서 태그만 뽑는다.


In [ ]:
PHASES = [
    {"id": "story", "energy": "low", "source": "source_story"},
    {"id": "case",  "energy": "mid", "source": "adapted_story"},
]

INSTRUMENT_OVERRIDE = {
    # "heungbu": ["daegeum", "haegeum", "geomungo", "janggu", "jing",
    #             "traditional korean", "gyemyeonjo"],
}

# 네거티브 프롬프트는 쓰지 않는다. GenerationParams 에 그런 필드가 없고,
# 보컬 차단은 instrumental=True + lyrics="[Instrumental]" 이 정식 방법이다.


def ace_tags(scen, d):
    era = d["setting"]["era"]
    out = {}
    for p in PHASES:
        tone = d[p["source"]]["tone"]
        tags = ["instrumental", "no vocals", "cinematic", "seamless loop",
                "sparse", f"{p['energy']} energy", tone, era]
        # scenarios_en 의 culture.instruments 는 이제 bgm.palette.reference_en 문장
        # 하나다(리스트가 아님) — 태그 하나로 통째로 붙인다. 없으면(예전
        # asset_input) INSTRUMENT_OVERRIDE(리스트)를 안전망으로 쓴다.
        instr = d.get("culture", {}).get("instruments")
        if instr:
            tags.append(instr)
        else:
            tags += INSTRUMENT_OVERRIDE.get(scen, [])
        out[p["id"]] = {"source": p["source"], "tone": tone, "tags": tags}
    return out


COND = {k: ace_tags(k, d) for k, d in DATA.items()}

N, SECONDS, TAG = 3, 60, ""   # SR 은 쓰지 않는다. 모델이 48kHz 로 직접 저장한다.

for k, ph in COND.items():
    print("=" * 70); print(k)
    for pid, v in ph.items():
        print(f"  [{pid}] <- {v['source']}")
        print(f"     {', '.join(v['tags'])}")

## 4. 생성

**SFT 체크포인트를 쓴다** — Turbo는 CFG를 지원하지 않아 네거티브 프롬프트가 무시된다.


In [ ]:
# ---------------------------------------------------------------------------
# ACE-Step 1.5 생성
#
# `acestep.pipeline_ace_step` / `ACEStepPipeline` 은 1.5 에 없다 (1.0 경로다).
# 아래는 공식 cli.py 의 초기화 순서를 그대로 옮긴 것이다:
#   AceStepHandler() -> get_available_acestep_v15_models() -> initialize_service()
#   LLMHandler()     -> thinking=False 면 초기화하지 않는다 (cli.py 도 동일)
#   generate_music(dit, llm, params, config, save_dir=...)
#
# 반환값은 오디오 배열이 아니라 '저장된 파일 경로'다. 그래서 sf.write 를 쓰지 않는다.
# 모델 출력은 48kHz 이고, 44100 으로 직접 쓰면 에러 없이 8.8% 느려지고 음이 낮아진다.
# ---------------------------------------------------------------------------
import pathlib
import acestep
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

ACE_DIR = pathlib.Path(acestep.__file__).resolve().parent.parent
print("ACE-Step 저장소:", ACE_DIR)

dit = AceStepHandler()
llm = LLMHandler()          # thinking=False 라 초기화하지 않는다

models = dit.get_available_acestep_v15_models()
if not models:
    print("체크포인트가 없다. 내려받는다 (오래 걸린다).")
    from acestep.model_downloader import ensure_main_model, get_checkpoints_dir
    ok, msg = ensure_main_model(get_checkpoints_dir())
    print(msg)
    models = dit.get_available_acestep_v15_models()

assert models, "DiT 모델을 찾지 못했다. acestep 체크포인트 다운로드를 확인할 것."
print("사용 가능한 모델:", models)

# turbo 는 CFG 를 지원하지 않는다. base 가 있으면 그쪽을 쓴다.
MODEL_CFG = next((m for m in models if "turbo" not in m.lower()), models[0])
IS_TURBO  = "turbo" in MODEL_CFG.lower()
print("선택:", MODEL_CFG, "(turbo — guidance_scale 무시됨)" if IS_TURBO else "")

dit.initialize_service(
    project_root=str(ACE_DIR),
    config_path=MODEL_CFG,
    device=DEVICE,
    use_flash_attention=dit.is_flash_attention_available(DEVICE),
    compile_model=False,
    offload_to_cpu=False,
    offload_dit_to_cpu=False,
)
print("핸들러 준비됨")

In [ ]:
STEPS      = 8 if IS_TURBO else 32     # turbo 8 / base 32~100
CFG_SCALE  = 7.0                       # turbo 에서는 무시된다

for scen, ph in COND.items():
    for pid, v in ph.items():
        outdir = OUT_DIR / scen / pid
        outdir.mkdir(parents=True, exist_ok=True)

        params = GenerationParams(
            task_type="text2music",
            caption=", ".join(v["tags"]),
            lyrics="[Instrumental]",
            instrumental=True,          # 보컬 차단. 네거티브 프롬프트보다 확실하다.
            duration=float(SECONDS),
            inference_steps=STEPS,
            guidance_scale=CFG_SCALE,
            seed=0,
            thinking=False,             # LLM CoT 미사용 -> LLMHandler 초기화 불필요
        )
        config = GenerationConfig(
            batch_size=N,
            use_random_seed=False,
            seeds=list(range(N)),
            audio_format="wav",         # 48kHz 로 저장된다
        )

        res = generate_music(dit, llm, params, config, save_dir=str(outdir))
        if not res.success:
            print("FAIL", scen, pid, "->", res.error)
            continue
        print("OK", scen, pid, f"({len(res.audios)}개)")
        for a in res.audios:
            print("    ", a.get("path", a) if isinstance(a, dict) else a)

print("\n저장 위치:", OUT_DIR)

In [ ]:
del dit, llm
free()

## 5. 두 모델 비교

같은 조건에서 나온 결과를 나란히 듣는다.


In [ ]:
from IPython.display import Audio, display
import glob

SA3_DIR = BASE / "outputs" / "bgm_stable_audio"


def compare(scen, phase):
    """두 모델의 같은 조건 결과를 나란히 듣는다.
       ACE 는 <시나리오>/<페이즈>/ 아래에 모델이 정한 이름으로 저장한다."""
    for label, pattern in [
        ("Stable Audio 3", str(SA3_DIR / scen / f"{phase}*.wav")),
        ("ACE-Step 1.5",   str(OUT_DIR / scen / phase / "*")),
    ]:
        fs = sorted(f for f in glob.glob(pattern)
                    if f.lower().endswith((".wav", ".flac", ".mp3")))
        print(f"--- {label} ({len(fs)}개) ---")
        for f in fs[:3]:
            print(f.split("/")[-1]); display(Audio(f))


demo_scen = next(iter(DATA))
compare(demo_scen, "case")